# PyTorch API 演示

每个常用 API 附带可运行示例，覆盖构造、形状变换、掩码、运算等类别。

In [1]:
import torch
import torch.nn.functional as F

---
## 一、构造 API

### `torch.tensor(data)`
直接从 Python 列表或 numpy 数组创建张量，会复制数据。

In [3]:
a = torch.tensor([1, 2, 3])
print("1D 整数张量:", a)

b = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("2D 浮点张量 shape:", b.shape, "\n", b)

c = torch.tensor([1, 2, 3], dtype=torch.float32)
print("指定 float32 dtype:", c, c.dtype)

1D 整数张量: tensor([1, 2, 3])
2D 浮点张量 shape: torch.Size([2, 2]) 
 tensor([[1., 2.],
        [3., 4.]])
指定 float32 dtype: tensor([1., 2., 3.]) torch.float32


### `torch.zeros(*size)` 与 `torch.zeros_like`
创建全 0 张量，常用于初始化偏置或掩码。

In [5]:
print(torch.zeros(3))        # 1D
print(torch.zeros(2, 3))     # 2D

x = torch.randn(2, 4)
print("zeros_like x:", torch.zeros_like(x))  # 与 x 同形状

tensor([0., 0., 0.])
tensor([[0., 0., 0.],
        [0., 0., 0.]])
zeros_like x: tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.]])


### `torch.ones(*size)` 与 `torch.ones_like`
创建全 1 张量，常用于初始化权重或掩码。

In [6]:
print(torch.ones(3))
print(torch.ones(2, 3))

x = torch.zeros(2, 4)
print("ones_like x:", torch.ones_like(x))

tensor([1., 1., 1.])
tensor([[1., 1., 1.],
        [1., 1., 1.]])
ones_like x: tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.]])


### `torch.arange(start, end, step)`
创建等差数列张量，类似 Python `range`，**不包含** end。

In [7]:
print(torch.arange(5))            # [0,1,2,3,4]
print(torch.arange(1, 5))         # [1,2,3,4]
print(torch.arange(0, 1, 0.2))    # [0.0, 0.2, 0.4, 0.6, 0.8]

tensor([0, 1, 2, 3, 4])
tensor([1, 2, 3, 4])
tensor([0.0000, 0.2000, 0.4000, 0.6000, 0.8000])


### `torch.randn(*size)`
从标准正态分布 N(0,1) 中随机采样，常用于参数初始化。

In [8]:
torch.manual_seed(42)
print(torch.randn(3))
print(torch.randn(2, 3))

x = torch.zeros(2, 3)
print("randn_like:", torch.randn_like(x))

tensor([0.3367, 0.1288, 0.2345])
tensor([[ 0.2303, -1.1229, -0.1863],
        [ 2.2082, -0.6380,  0.4617]])
randn_like: tensor([[ 0.2674,  0.5349,  0.8094],
        [ 1.1103, -1.6898, -0.9890]])


### `torch.empty(*size)`
创建**未初始化**张量，内存中原有什么值就是什么值。比 zeros/ones 快，必须后续赋值才能安全使用。

In [9]:
e = torch.empty(3)
print("empty(3) 原始值（不确定）:", e)

e2 = torch.empty(2, 3)
e2.fill_(0)   # 后续用 fill_ 显式赋值
print("fill_(0) 后:", e2)

empty(3) 原始值（不确定）: tensor([0., 0., 0.])
fill_(0) 后: tensor([[0., 0., 0.],
        [0., 0., 0.]])


---
## 二、形状变换 API

### `x.view(*shape)`
重塑形状，**要求内存连续**。不复制数据，只改变解释方式。

In [10]:
x = torch.arange(6)      # shape (6,)
print("原始:", x, x.shape)
print("view(2,3):\n", x.view(2, 3))
print("view(3,-1):\n", x.view(3, -1))   # -1 自动推断

原始: tensor([0, 1, 2, 3, 4, 5]) torch.Size([6])
view(2,3):
 tensor([[0, 1, 2],
        [3, 4, 5]])
view(3,-1):
 tensor([[0, 1],
        [2, 3],
        [4, 5]])


### `x.reshape(*shape)`
重塑形状，**不要求内存连续**，比 view 更安全（可能返回视图也可能复制数据）。

In [ ]:
x = torch.arange(6)
print(x.reshape(2, 3))
print(x.reshape(-1))    # 展平为 1D

### `x.transpose(dim0, dim1)`
交换两个指定维度。

In [ ]:
x = torch.arange(6).reshape(2, 3).float()
print("原始 (2,3):\n", x)
print("transpose(0,1) -> (3,2):\n", x.transpose(0, 1))

y = torch.randn(2, 3, 4)
print("3D transpose(1,2): shape", y.transpose(1, 2).shape)  # (2,4,3)

### `x.permute(*dims)`
按指定顺序重排**所有维度**，是 transpose 的泛化版本。

In [ ]:
x = torch.randn(2, 3, 4)   # (batch, seq, dim)
print("原始 shape:", x.shape)
print("permute(0,2,1) -> (batch, dim, seq):", x.permute(0, 2, 1).shape)
print("permute(2,0,1):", x.permute(2, 0, 1).shape)

### `x.unsqueeze(dim)`
在指定位置插入大小为 1 的维度，常用于扩充 batch 或 channel 维。

In [ ]:
x = torch.tensor([1, 2, 3])    # shape (3,)
print("原始:", x.shape)
print("unsqueeze(0) -> (1,3):", x.unsqueeze(0).shape, x.unsqueeze(0))
print("unsqueeze(1) -> (3,1):", x.unsqueeze(1).shape)
print("unsqueeze(-1) -> (3,1):", x.unsqueeze(-1).shape)

### `x.squeeze(dim)`
删除大小为 1 的维度。不指定 dim 则删除所有 size=1 的维度。

In [ ]:
x = torch.randn(1, 3, 1, 4)
print("原始 shape:", x.shape)
print("squeeze() 全删:", x.squeeze().shape)    # (3, 4)
print("squeeze(0):", x.squeeze(0).shape)        # (3, 1, 4)
print("squeeze(2):", x.squeeze(2).shape)        # (1, 3, 4)

---
## 三、掩码 API

### `torch.triu(input, diagonal=0)`
取矩阵**上三角**部分，其余置 0。diagonal > 0 右移，< 0 左移。

In [ ]:
x = torch.ones(4, 4)
print("上三角（含主对角线）:\n", torch.triu(x))
print("严格上三角（不含主对角线，常用于 causal mask）:\n", torch.triu(x, diagonal=1))

### `torch.tril(input, diagonal=0)`
取矩阵**下三角**部分，其余置 0。

In [ ]:
x = torch.ones(4, 4)
print("下三角（含主对角线）:\n", torch.tril(x))
print("严格下三角（不含主对角线）:\n", torch.tril(x, diagonal=-1))

### `tensor.masked_fill(mask, value)`
将 mask 中为 `True` 的位置填充为指定 value。常与 triu/tril 配合实现注意力掩码。

In [ ]:
torch.manual_seed(0)
x = torch.randn(4, 4)
# 严格上三角为 True（未来位置）
mask = torch.triu(torch.ones(4, 4), diagonal=1).bool()
print("mask:\n", mask)

# 填充 -inf，softmax 后变 0，实现 causal attention
out = x.masked_fill(mask, float('-inf'))
print("masked_fill(-inf) 后:\n", out)
print("对行做 softmax 后:\n", F.softmax(out, dim=-1).round(decimals=3))

### 比较运算符 `==, !=, >, <`
逐元素比较，返回 bool 型张量，常用于生成 padding mask 等。

In [ ]:
x = torch.tensor([1, 2, 3, 0, 0])
print("x == 0  (padding mask):", x == 0)
print("x > 1              :", x > 1)
print("x != 0             :", x != 0)

---
## 四、运算 API

### `tensor.sum(dim, keepdim)`
沿指定维度求和。

In [ ]:
x = torch.tensor([[1., 2., 3.], [4., 5., 6.]])   # shape (2, 3)
print("全局 sum:", x.sum())
print("dim=0 按列求和:", x.sum(dim=0))            # shape (3,)
print("dim=1 按行求和:", x.sum(dim=1))            # shape (2,)
print("dim=1 keepdim:", x.sum(dim=1, keepdim=True))  # shape (2,1)

### `tensor.mean(dim, keepdim)`
沿指定维度求均值，用法同 sum。

In [ ]:
x = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
print("全局 mean:", x.mean())
print("dim=1 每行均值:", x.mean(dim=1))
print("dim=1 keepdim:", x.mean(dim=1, keepdim=True))

### `tensor.max(dim, keepdim)`
不指定 dim 返回全局最大值标量；指定 dim 返回 `(values, indices)` 命名元组。

In [ ]:
x = torch.tensor([[1., 3., 2.], [5., 4., 6.]])
print("全局 max:", x.max())

values, indices = x.max(dim=1)
print("dim=1 每行最大值:", values)
print("dim=1 最大值列索引:", indices)

### `torch.nn.functional.softmax(x, dim)`
对指定维度做 softmax，将数值转为概率分布（和为 1）。

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
prob = F.softmax(x, dim=0)
print("softmax:", prob)
print("和为 1:", prob.sum())

# 注意力 score -> attention weight
torch.manual_seed(0)
scores = torch.randn(2, 4, 4)   # (batch, seq, seq)
attn = F.softmax(scores, dim=-1)
print("\n注意力权重（每行和为1）:\n", attn[0].round(decimals=3))

### `tensor.argmax(dim)`
返回指定维度上最大值的**索引**，常用于分类任务取预测类别。

In [ ]:
x = torch.tensor([[1., 3., 2.], [5., 4., 6.]])
print("全局 argmax（展平索引）:", x.argmax())
print("dim=1 每行最大值列索引:", x.argmax(dim=1))

# 分类场景
logits = torch.tensor([[0.1, 2.5, 0.3], [1.2, 0.5, 3.1]])
pred = logits.argmax(dim=-1)
print("预测类别:", pred)   # [1, 2]

---
## 五、torch 工具 API

### `torch.clamp(input, min, max)`
将张量数值限制在 [min, max] 范围内，超出部分截断到边界。

In [ ]:
x = torch.tensor([-1., 0., 0.5, 2., 3.])
print("clamp(0, 1)  :", torch.clamp(x, 0, 1))
print("clamp(min=0) :", torch.clamp(x, min=0))   # 只限下界
print("clamp(max=1) :", torch.clamp(x, max=1))   # 只限上界

# 实际场景：保证网络输出 >= 1（如预测行人数量）
fake_output = torch.tensor([-0.5, 0.8, 1.2, 3.0])
clipped = torch.clamp(fake_output, 1, float('inf'))
print("预测值限制 >= 1:", clipped)

---
## 六、Tensor 扩展 API

### `tensor.repeat(*sizes)`
沿每个维度复制数据指定次数，会分配新内存（区别于 expand 的零拷贝）。

In [ ]:
x = torch.tensor([1, 2, 3])         # shape (3,)
print("repeat(2):", x.repeat(2))    # [1,2,3,1,2,3]

x2 = torch.tensor([[1, 2, 3]])       # shape (1, 3)
print("repeat(2,1):\n", x2.repeat(2, 1))   # (2,3)
print("repeat(2,3):\n", x2.repeat(2, 3))   # (2,9)

### `tensor.expand(*sizes)`
将 size 为 1 的维度广播扩展到指定大小，**零拷贝**，比 repeat 更高效。

In [ ]:
x = torch.tensor([[1], [2], [3]])    # shape (3, 1)
print("原始 shape:", x.shape)
expanded = x.expand(3, 4)
print("expand(3,4):\n", expanded)
print("expand(-1,4) 等价（-1=保持不变）:\n", x.expand(-1, 4))

# expand 不复制数据，与 repeat 对比
print("expand 存储 id 相同:", expanded.storage().data_ptr() == x.storage().data_ptr())

### `tensor.contiguous()`
返回内存**连续存储**的张量副本。transpose/permute 后内存不连续，需先调用此方法再使用 view。

In [ ]:
x = torch.randn(2, 3)
y = x.transpose(0, 1)                  # transpose 后内存不连续
print("transpose 后 is_contiguous:", y.is_contiguous())

# 直接 view 会报错，需先 contiguous
z = y.contiguous()
print("contiguous 后 is_contiguous:", z.is_contiguous())
print("contiguous().view(-1) shape:", z.view(-1).shape)

# reshape 内部自动处理，更安全
print("reshape(-1) shape:", y.reshape(-1).shape)